# 더미 데이터 만들기 (연체 패턴 포함)
- 목적 : 연체와 관련된 패턴을 일부러 넣은 데이터로 `ml_solution.ipynb` 를 다시 돌려, 모델이 패턴을 찾아내는지 확인
- 방법 : `clean_rentals.csv` 는 그대로 두고, 규칙에 따라 **반납 시각만 다시 생성** → `data/dummy_rentals.csv`

## [1] 원본 불러오기 & 규칙에 쓸 값
- `hour`, `weekday`, `member_days` (음수는 0)

In [1]:
import pandas as pd
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)      # 무작위 숫자 생성기 (시드 고정)

# 1. 원본 불러오기 + 결합
df = pd.read_csv("data/clean_rentals.csv", parse_dates=["rent_time", "return_time"])
users = pd.read_csv("data/users.csv", parse_dates=["join_date"])
df = df.merge(users, on="user_id", how="left", validate="many_to_one")

# 2. 규칙에 쓸 값들
df["hour"] = df["rent_time"].dt.hour
df["weekday"] = df["rent_time"].dt.weekday
df["member_days"] = (df["rent_time"] - df["join_date"]).dt.days.clip(lower=0)

print(f"행 수 : {len(df):,}")
display(df[["rent_time", "hour", "weekday", "member_days", "membership_type"]].head())

행 수 : 13,501


,rent_time,hour,weekday,member_days,membership_type
0,2022-04-18 20:28:00,20,0,370,BASIC
1,2022-05-19 09:53:00,9,3,0,BASIC
2,2022-09-07 16:20:00,16,2,344,BASIC
3,2022-10-11 06:39:00,6,1,542,BASIC
4,2022-07-11 10:15:00,10,0,485,BASIC


## [2] 연체 확률 규칙
| 규칙 | 확률 변화 |
|:--|:--:|
| 기본 | 10% |
| 20시 이후 대여 | +25% |
| 주말(토, 일) | +10% |
| 가입 1~30일 신규 | +20% |
| 프리미엄 멤버십 | -7% |

- 전체 평균 약 16% (원래 데이터 17.5% 와 비슷하게)

In [2]:
# 3. 행마다 연체 확률 (규칙)
p = (0.10
     + np.where(df["hour"] >= 20, 0.25, 0)                      # 밤 대여
     + np.where(df["weekday"] >= 5, 0.10, 0)                    # 주말 (5=토, 6=일)
     + np.where(df["member_days"].between(1, 30), 0.20, 0)      # 신규 이용자
     + np.where(df["membership_type"] == "PREMIUM", -0.07, 0))  # 프리미엄

df["p_overdue"] = p

print(f"평균 연체 확률 : {p.mean():.1%}")
print(f"최소 {p.min():.0%} ~ 최대 {p.max():.0%}")

평균 연체 확률 : 16.0%
최소 3% ~ 최대 65%


## [3] 확률대로 연체 여부 뽑기
- 0~1 사이 무작위 숫자 < 연체 확률 → 연체(1)

In [8]:
# 4. 확률대로 연체 여부 뽑기
u = rng.random(len(df))                              # 0 ~ 1 사이 무작위 숫자를 행 수만큼
df["is_overdue"] = (u < df["p_overdue"]).astype(int)

print(f"연체 : {df['is_overdue'].sum():,}건 ({df['is_overdue'].mean():.1%})")

# 5. 대여 시간(분) 다시 만들기
normal_min = rng.integers(10, 101, size=len(df))     # 10 ~ 100분
overdue_min = rng.integers(101, 181, size=len(df))   # 101 ~ 180분

df["duration_min"] = np.where(df["is_overdue"] == 1, overdue_min, normal_min)
df["return_time"] = df["rent_time"] + pd.to_timedelta(df["duration_min"], unit="min")

print("[연체 여부별 대여 시간]")
display(df.groupby("is_overdue")["duration_min"].agg(["min", "max", "mean"]).round(1))

연체 : 2,189건 (16.2%)
[연체 여부별 대여 시간]


,min,max,mean
is_overdue,,,
0,10,100,55.1
1,101,180,140.4


In [9]:
# 6. 요금 : 대여 시간 × 분당 요금
per_min = {"일반": 20, "전동": 30}
df["fee"] = (df["duration_min"] * df["bike_type"].map(per_min)).round(-1).astype(int)

# 7. 거리 : 속도(km/h) × 시간(시)
speed = np.where(df["bike_type"] == "전동",
                 rng.uniform(15, 25, size=len(df)),
                 rng.uniform(8, 15, size=len(df)))
df["distance_km"] = (speed * df["duration_min"] / 60).round(1)

print("[자전거 타입별 평균]")
display(df.groupby("bike_type")[["duration_min", "fee", "distance_km"]].mean().round(1))

[자전거 타입별 평균]


,duration_min,fee,distance_km
bike_type,,,
일반,69.2,1384.0,13.2
전동,67.7,2032.4,22.7


## [6] 저장
- 규칙 계산용으로 만든 컬럼은 빼고, `clean_rentals.csv` 와 **같은 컬럼**만 저장 → `data/dummy_rentals.csv`

In [10]:
# 8. 원래 컬럼만 남겨서 저장
orig_cols = pd.read_csv("data/clean_rentals.csv", nrows=0).columns   # 원래 컬럼 이름만 읽기
dummy = df[orig_cols]

dummy.to_csv("data/dummy_rentals.csv", index=False, encoding="utf-8-sig")

print(f"저장 : {dummy.shape}")
print(f"컬럼 같음 : {list(dummy.columns) == list(orig_cols)}")

저장 : (13501, 16)
컬럼 같음 : True
